In [0]:
files = {
    "employee" : "/Volumes/workspace/default/assignment/employees_extended.json",
    "Transaction" : "/Volumes/workspace/default/assignment/transactions_1200 (1).json"
}
df = {}

for name, path in files.items():
    df[name]=spark.read.json(
        path
    )

In [0]:
from pyspark.sql.functions import col

df["Transaction"].display()

amount,emp_id,location,payment_mode,txn_date,txn_id
4450.62,275,"List(Chennai, 614116)",Cash,2024-04-14,1
19018.3,40,"List(Pune, 644669)",Cash,2025-02-25,2
8067.15,54,"List(Pune, 643751)",Wallet,2023-11-22,3
8272.46,252,"List(Hyderabad, 678328)",Card,2024-06-12,4
429.09,72,"List(Delhi, 605752)",Card,2024-07-07,5
7194.54,684,"List(Chennai, 656444)",Cash,2022-02-13,6
13339.55,489,"List(Kochi, 645018)",Cash,2022-03-17,7
10422.44,495,"List(Kochi, 674031)",Wallet,2025-06-01,8
726.15,323,"List(Pune, 656935)",Wallet,2025-08-16,9
31953.83,552,"List(Hyderabad, 663718)",Wallet,2025-09-27,10


### Q9: Join with Transactions

In [0]:
df["employee"].join(
    df["Transaction"],
    df["employee"]["emp_id"] == df["Transaction"]["emp_id"]
).select("name","department","amount").display()

name,department,amount
Ravi K,Operations,20826.44
Anu V,IT,25313.96
Kiran N,Operations,26267.69
Divya B,Support,5194.22
Vikram V,Sales,15669.01
Ravi J,Finance,22990.61
Karthik M,Marketing,32704.51
Manoj V,Support,426.44
Ravi H,Operations,11615.6
John J,Finance,39862.69


### Q10: Department Revenue 

In [0]:
from pyspark.sql.functions import expr

df["employee"] = df["employee"].withColumn(
    "amount",
    expr("try_cast(amount as float)")
    )

In [0]:
df["Transaction"].printSchema()

root
 |-- amount: double (nullable = true)
 |-- emp_id: long (nullable = true)
 |-- location: struct (nullable = true)
 |    |-- city: string (nullable = true)
 |    |-- pincode: string (nullable = true)
 |-- payment_mode: string (nullable = true)
 |-- txn_date: string (nullable = true)
 |-- txn_id: long (nullable = true)



In [0]:
from pyspark.sql.functions import sum

e = df["employee"]
t = df["Transaction"]

joined = e.join(
    t,
    "emp_id",
    "inner"
)

joined.display()

emp_id,address,age,department,is_active,join_date,metadata,name,projects,salary,skills,amount,location,payment_mode,txn_date,txn_id
1,"List(Hyderabad, TS)",26,Operations,false,2024-02-12,"List(L3, null, manual)",Ravi K,"List(List(35, Alpha), List(131, Alpha), List(278, Beta))",59378.66,List(GCP),20826.44,"List(Delhi, 654684)",Card,2023-05-07,101
3,"List(Hyderabad, TS)",55,IT,true,2025-01-18,"List(null, null, api)",Anu V,List(),146639.47,"List(GCP, Scala, SQL)",25313.96,"List(Pune, 608947)",NetBanking,2025-01-25,1115
4,"List(Delhi, DL)",42,Operations,true,2020-12-29,"List(L3, null, api)",Kiran N,"List(List(393, Epsilon))",null,List(GCP),26267.69,"List(Hyderabad, 656968)",Cash,2025-03-20,1087
5,"List(Hyderabad, TS)",22,Support,true,2022-07-02,"List(null, IN, null)",Divya B,"List(List(355, Beta), List(222, Delta))",121999.99,"List(SQL, Scala, GCP)",5194.22,"List(Hyderabad, 683546)",NetBanking,2023-01-05,923
6,"List(Hyderabad, TS)",57,Sales,false,2020-06-17,"List(null, IN, null)",Vikram V,List(),118376.73,"List(Azure, AWS)",15669.01,"List(Chennai, 687262)",NetBanking,2023-06-18,292
7,"List(Kochi, KL)",36,Finance,false,2018-02-17,"List(L3, null, ingest)",Ravi J,"List(List(348, Omega), List(77, Gamma))",99486.05,List(Tableau),22990.61,"List(Mumbai, 637708)",Wallet,2025-05-28,401
8,"List(Kochi, KL)",39,Marketing,false,2023-09-11,"List(null, null, null)",Karthik M,"List(List(418, Epsilon))",139078.48,"List(Spark, Excel)",32704.51,"List(Pune, 649656)",Cash,2022-02-10,1069
10,"List(Bangalore, KA)",39,Support,true,2025-07-14,"List(L3, null, ingest)",Manoj V,List(),116241.38,"List(AWS, Azure, Airflow)",426.44,"List(Pune, 678752)",NetBanking,2023-11-16,1013
11,"List(Chennai, TN)",22,Operations,false,2018-10-18,"List(null, IN, api)",Ravi H,"List(List(296, Beta), List(390, Beta), List(315, Epsilon))",null,"List(Java, Tableau)",11615.6,"List(Hyderabad, 615751)",Wallet,2022-11-10,1027
12,"List(Mumbai, MH)",null,Finance,false,2021-10-21,"List(L2, IN, api)",John J,"List(List(294, Beta))",null,List(Scala),39862.69,"List(Chennai, 680736)",Card,2025-08-13,1058


### Q11: Top Performer per Department 

In [0]:
from pyspark.sql.functions import sum

e = df["employee"]
t = df["Transaction"]

e.join(
    t,
    "emp_id"
).groupBy(
    "department"
).agg(
    sum("amount").alias("total_revenue")
).display()

### Q12: Left Join Scenario 

In [0]:
e = df["employee"]
t = df["Transaction"]

e.join(
    t,
    "emp_id",
    "left_anti"
).select("emp_id","name").display()

emp_id,name
2,Anu S
9,Ravi C
23,John R
27,Priya H
28,Aisha M
31,Priya J
32,Meena N
50,Ravi B
64,Ravi B
68,Anu J


In [0]:
from pyspark.sql.functions import col,when, row_number 
from pyspark.sql.window import Window

total_Sales = df["Transaction"].agg(sum("amount").alias("total_revenue"))

e = df["employee"]
t = df["Transaction"]

joined = e.join(
    t,
    "emp_id"
)


window_spec = Window.partitionBy("department").orderBy(col("amount").desc())

top_emp = joined.withColumn(
    "rn",
    row_number().over(window_spec)
)

top_emp.filter(
    col("rn")==1
).select(
    "department","name","amount"
).display()

